# PageRank on the Thai Wikipedia link graph: CPU vs GPU (CUDA)

**Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.

1. Check the GPU
2. (Optional) Google Drive cache, so the dump is downloaded only once
3. Download the Thai Wikipedia dump and convert it to a graph (`prep/convert.py`)
4. Compile `pagerank.cu` with `nvcc`
5. Correctness run on the full graph and the top-20 Thai pages
6. Benchmark CPU vs three GPU versions on 10 / 25 / 50 / 100 % of the graph
7. Charts for the slides

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!nvcc --version | tail -2

## 1. Source files (same as the GitHub repo)

In [ ]:
!mkdir -p prep data dump results

In [ ]:
%%writefile prep/convert.py
#!/usr/bin/env python3
"""
convert.py - turn a Thai Wikipedia SQL dump into a PageRank-ready graph.

Input  (from https://dumps.wikimedia.org/thwiki/latest/):
    thwiki-latest-page.sql.gz        every page: id, namespace, title, is_redirect
    thwiki-latest-redirect.sql.gz    redirect page -> target title
    thwiki-latest-pagelinks.sql.gz   link: from page -> target
    thwiki-latest-linktarget.sql.gz  (newer dumps only) target id -> title

Output (in --out directory):
    graph_100.bin    full graph in CSR format, incoming links (see write_graph)
    graph_50.bin, graph_25.bin, graph_10.bin   smaller graphs for the speedup-vs-size chart
    titles.txt       title of vertex i on line i (UTF-8, Thai)

Usage:
    python3 convert.py --dump DIR --out DIR
"""
import argparse, gzip, os, re, struct, sys, time
from array import array
import numpy as np

# ---------------------------------------------------------------------------
# 1. Reading MySQL dump files
# ---------------------------------------------------------------------------
# A dump is a .sql.gz file containing:
#   CREATE TABLE `page` ( `page_id` int ..., `page_namespace` int ..., ... );
#   INSERT INTO `page` VALUES (1,0,'Title',0,...),(2,0,'Other',1,...),...;
# We read the CREATE TABLE to learn the column order (so we do not depend on
# one fixed schema), then stream the INSERT lines tuple by tuple.

VALUE_RE = re.compile(r"-?\d+(?:\.\d+)?(?:e-?\d+)?|NULL|'(?:[^'\\]|\\.)*'", re.S)
TUPLE_RE = re.compile(
    r"\(((?:-?\d+(?:\.\d+)?(?:e-?\d+)?|NULL|'(?:[^'\\]|\\.)*')"
    r"(?:,(?:-?\d+(?:\.\d+)?(?:e-?\d+)?|NULL|'(?:[^'\\]|\\.)*'))*)\)", re.S)
ESCAPES = {"0": "\0", "n": "\n", "r": "\r", "t": "\t", "Z": "\x1a",
           "\\": "\\", "'": "'", '"': '"'}

def unquote(v):
    """Turn one SQL value into a Python int, str or None."""
    if v == "NULL":
        return None
    if v[0] == "'":
        s = v[1:-1]
        if "\\" in s:
            s = re.sub(r"\\(.)", lambda m: ESCAPES.get(m.group(1), m.group(1)), s)
        return s
    return int(v) if re.fullmatch(r"-?\d+", v) else float(v)

def read_table(path, wanted):
    """Yield dicts {column: value} for the columns in `wanted`, row by row."""
    columns, idx = None, None
    with gzip.open(path, "rt", encoding="utf-8", errors="replace") as f:
        in_create = False
        for line in f:
            if line.startswith("CREATE TABLE"):
                in_create, columns = True, []
                continue
            if in_create:
                m = re.match(r"\s*`(\w+)`", line)
                if m:
                    columns.append(m.group(1))
                if line.startswith(")"):
                    in_create = False
                    missing = [c for c in wanted if c not in columns]
                    if missing:
                        raise SystemExit(f"{path}: columns {missing} not found; has {columns}")
                    idx = [columns.index(c) for c in wanted]
                continue
            # Old dumps put a whole INSERT (thousands of rows) on one line;
            # newer dumps write "INSERT INTO ... VALUES" and then one "(row)," per line.
            if not (line.startswith("INSERT INTO") or line.startswith("(")):
                continue
            for t in TUPLE_RE.finditer(line):
                vals = VALUE_RE.findall(t.group(1))
                yield {c: unquote(vals[i]) for c, i in zip(wanted, idx)}

def columns_of(path):
    """Return the column names declared in a dump's CREATE TABLE."""
    cols = []
    with gzip.open(path, "rt", encoding="utf-8", errors="replace") as f:
        inside = False
        for line in f:
            if line.startswith("CREATE TABLE"):
                inside = True
                continue
            if inside:
                m = re.match(r"\s*`(\w+)`", line)
                if m:
                    cols.append(m.group(1))
                if line.startswith(")"):
                    return cols
    return cols

# ---------------------------------------------------------------------------
# 2. Building the graph
# ---------------------------------------------------------------------------
def build(dump_dir, prefix):
    p = lambda name: os.path.join(dump_dir, f"{prefix}-{name}.sql.gz")
    t0 = time.time()

    # 2a. Pages: keep namespace 0 (real articles), remember which are redirects.
    title_to_id, is_redirect, id_to_title = {}, set(), {}
    for r in read_table(p("page"), ["page_id", "page_namespace", "page_title", "page_is_redirect"]):
        if r["page_namespace"] != 0:
            continue
        pid, title = r["page_id"], r["page_title"]
        title_to_id[title] = pid
        id_to_title[pid] = title
        if r["page_is_redirect"]:
            is_redirect.add(pid)
    print(f"pages (namespace 0): {len(id_to_title):,}  redirects: {len(is_redirect):,}  [{time.time()-t0:.0f}s]")

    # 2b. Redirects: redirect page id -> final article id (follow one hop).
    redirect_to = {}
    for r in read_table(p("redirect"), ["rd_from", "rd_namespace", "rd_title"]):
        if r["rd_from"] in is_redirect and r["rd_namespace"] == 0:
            tgt = title_to_id.get(r["rd_title"])
            if tgt is not None and tgt not in is_redirect:
                redirect_to[r["rd_from"]] = tgt

    def resolve(pid):
        """Article id a link really points to, or None (missing / broken redirect)."""
        if pid is None:
            return None
        if pid in is_redirect:
            return redirect_to.get(pid)
        return pid

    # 2c. Articles = namespace-0 pages that are not redirects. Number them 0..V-1.
    articles = sorted(pid for pid in id_to_title if pid not in is_redirect)
    vid = {pid: i for i, pid in enumerate(articles)}
    V = len(articles)

    # 2d. Links. Newer dumps store the target as pl_target_id -> linktarget table;
    #     older dumps store pl_namespace + pl_title directly. Support both.
    pl_cols = columns_of(p("pagelinks"))
    src, dst = array("i"), array("i")      # compact int32 arrays, not Python lists
    if "pl_target_id" in pl_cols:
        target_of = {}  # linktarget id -> article vertex
        for r in read_table(p("linktarget"), ["lt_id", "lt_namespace", "lt_title"]):
            if r["lt_namespace"] == 0:
                a = resolve(title_to_id.get(r["lt_title"]))
                if a is not None:
                    target_of[r["lt_id"]] = vid[a]
        rows = read_table(p("pagelinks"), ["pl_from", "pl_from_namespace", "pl_target_id"])
        get_target = lambda r: target_of.get(r["pl_target_id"])
    else:
        rows = read_table(p("pagelinks"), ["pl_from", "pl_from_namespace", "pl_namespace", "pl_title"])
        def get_target(r):
            if r["pl_namespace"] != 0:
                return None
            a = resolve(title_to_id.get(r["pl_title"]))
            return None if a is None else vid[a]

    for r in rows:
        if r["pl_from_namespace"] != 0:
            continue
        u = vid.get(r["pl_from"])          # source must be a real article
        if u is None:
            continue
        v = get_target(r)
        if v is None or v == u:            # drop missing targets and self-links
            continue
        src.append(u); dst.append(v)

    src = np.frombuffer(src, dtype=np.int32).astype(np.int64)
    dst = np.frombuffer(dst, dtype=np.int32).astype(np.int64)
    # Remove duplicate links (same page linked twice, or via two redirects).
    key = np.unique(src * V + dst)
    src, dst = (key // V).astype(np.int32), (key % V).astype(np.int32)
    titles = [id_to_title[pid] for pid in articles]
    print(f"articles V = {V:,}  links E = {len(src):,}  [{time.time()-t0:.0f}s]")
    return V, src, dst, titles

# ---------------------------------------------------------------------------
# 3. CSR of incoming links + writing the binary file
# ---------------------------------------------------------------------------
def write_graph(path, V, src, dst):
    """
    Binary layout (little-endian):
        int32 V, int32 E
        int32 row_ptr[V+1]   incoming links of v are col_idx[row_ptr[v] .. row_ptr[v+1]-1]
        int32 col_idx[E]     the source page of each incoming link
        int32 out_deg[V]     number of outgoing links of each page (0 = dangling)
    """
    order = np.argsort(dst, kind="stable")          # group links by destination
    col_idx = src[order].astype(np.int32)
    counts = np.bincount(dst, minlength=V)
    row_ptr = np.zeros(V + 1, dtype=np.int32)
    np.cumsum(counts, out=row_ptr[1:])
    out_deg = np.bincount(src, minlength=V).astype(np.int32)
    with open(path, "wb") as f:
        f.write(struct.pack("<ii", V, len(col_idx)))
        f.write(row_ptr.tobytes()); f.write(col_idx.tobytes()); f.write(out_deg.tobytes())
    dangling = int((out_deg == 0).sum())
    print(f"  {os.path.basename(path)}: V={V:,} E={len(col_idx):,} dangling={dangling:,} "
          f"max in-degree={int(counts.max()) if V else 0:,}")

def subgraph(V, src, dst, frac, seed=42):
    """Induced subgraph on a random `frac` of the vertices (same seed = same result)."""
    rng = np.random.default_rng(seed)
    keep = np.sort(rng.choice(V, size=max(1, int(V * frac)), replace=False))
    new_id = np.full(V, -1, dtype=np.int64); new_id[keep] = np.arange(len(keep))
    m = (new_id[src] >= 0) & (new_id[dst] >= 0)
    return len(keep), new_id[src[m]].astype(np.int32), new_id[dst[m]].astype(np.int32)

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--dump", required=True, help="folder with the .sql.gz files")
    ap.add_argument("--out", required=True, help="output folder")
    ap.add_argument("--prefix", default="thwiki-latest")
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)

    V, src, dst, titles = build(a.dump, a.prefix)
    with open(os.path.join(a.out, "titles.txt"), "w", encoding="utf-8") as f:
        f.write("\n".join(t.replace("_", " ") for t in titles) + "\n")
    write_graph(os.path.join(a.out, "graph_100.bin"), V, src, dst)
    for pct in (50, 25, 10):
        sv, ss, sd = subgraph(V, src, dst, pct / 100)
        write_graph(os.path.join(a.out, f"graph_{pct}.bin"), sv, ss, sd)

if __name__ == "__main__":
    main()

In [ ]:
%%writefile pagerank.cu
// pagerank.cu - PageRank on the Thai Wikipedia link graph, CPU vs GPU.
//
// Build:  nvcc -O3 -arch=sm_75 -o pagerank pagerank.cu      (sm_75 = Colab T4)
// Run:    ./pagerank data/graph_100.bin [data/titles.txt | -] [fixed_iters]
//         fixed_iters > 0 runs exactly that many iterations on CPU and GPU
//         (same work on both = fair timing); otherwise stop when converged.
//
// Graph file (written by prep/convert.py), all little-endian int32:
//     V, E, row_ptr[V+1], col_idx[E], out_deg[V]
// row_ptr/col_idx is CSR of INCOMING links: the pages that link to v are
// col_idx[row_ptr[v]] .. col_idx[row_ptr[v+1]-1].
//
// PageRank (d = 0.85), one iteration:
//     contrib[u] = rank[u] / out_deg[u]            (0 for dangling pages)
//     D          = sum of rank[u] over dangling u  (their rank is spread over all pages)
//     new[v]     = (1-d)/V + d * ( sum over u->v of contrib[u]  +  D/V )
// Repeat until sum |new - rank| < TOL or MAX_ITER iterations.

#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <math.h>
#include <time.h>
#ifndef CPU_ONLY
#include <cuda_runtime.h>
#endif

#define DAMPING   0.85f
#define TOL       1e-5      // float sums of ~10^4 terms cannot go much lower
#define MAX_ITER  100
#define BLOCK_SIZE 256
#define TOP_K     20

#ifndef CPU_ONLY
#define CHECK(call)                                                          \
    do {                                                                     \
        cudaError_t err = (call);                                            \
        if (err != cudaSuccess) {                                            \
            printf("CUDA error %s at %s:%d\n", cudaGetErrorString(err),      \
                   __FILE__, __LINE__);                                      \
            exit(1);                                                         \
        }                                                                    \
    } while (0)
#endif

// ---------------------------------------------------------------------------
// Graph loading
// ---------------------------------------------------------------------------
typedef struct {
    int V, E;
    int *row_ptr;   // V+1
    int *col_idx;   // E   source of each incoming link
    int *out_deg;   // V
} Graph;

static Graph load_graph(const char *path)
{
    Graph g;
    FILE *f = fopen(path, "rb");
    if (!f) { printf("cannot open %s\n", path); exit(1); }
    if (fread(&g.V, 4, 1, f) != 1 || fread(&g.E, 4, 1, f) != 1) { printf("bad file\n"); exit(1); }
    g.row_ptr = (int *)malloc((size_t)(g.V + 1) * sizeof(int));
    g.col_idx = (int *)malloc((size_t)g.E * sizeof(int));
    g.out_deg = (int *)malloc((size_t)g.V * sizeof(int));
    if (fread(g.row_ptr, 4, g.V + 1, f) != (size_t)(g.V + 1) ||
        fread(g.col_idx, 4, g.E, f) != (size_t)g.E ||
        fread(g.out_deg, 4, g.V, f) != (size_t)g.V) { printf("truncated file\n"); exit(1); }
    fclose(f);
    return g;
}

static double now_ms(void)
{
    struct timespec t;
    clock_gettime(CLOCK_MONOTONIC, &t);
    return t.tv_sec * 1e3 + t.tv_nsec / 1e6;
}

// ---------------------------------------------------------------------------
// Sequential PageRank (the CPU baseline)
// ---------------------------------------------------------------------------
// One iteration reads every vertex and every edge exactly once: O(V + E).
// Any correct algorithm must read every edge at least once per iteration
// (each edge changes a rank), so this is optimal per iteration.
// Pull order (loop over v, read its incoming list) writes each new[v] once,
// sequentially, and precomputing contrib[] replaces E divisions by V.
static int pagerank_cpu(const Graph *g, float *rank, int fixed)
{
    int V = g->V;
    float *contrib = (float *)malloc(V * sizeof(float));
    float *next = (float *)malloc(V * sizeof(float));
    float base = (1.0f - DAMPING) / V;

    for (int v = 0; v < V; v++)
        rank[v] = 1.0f / V;

    int maxit = fixed > 0 ? fixed : MAX_ITER;
    int iter;
    for (iter = 1; iter <= maxit; iter++) {
        // map: contrib, and reduce: dangling mass
        double dangling = 0.0;
        for (int u = 0; u < V; u++) {
            if (g->out_deg[u] > 0)
                contrib[u] = rank[u] / g->out_deg[u];
            else {
                contrib[u] = 0.0f;
                dangling += rank[u];
            }
        }
        float dterm = DAMPING * (float)(dangling / V);

        // pull: gather incoming contributions, and reduce: L1 change
        double diff = 0.0;
        for (int v = 0; v < V; v++) {
            float sum = 0.0f;
            for (int j = g->row_ptr[v]; j < g->row_ptr[v + 1]; j++)
                sum += contrib[g->col_idx[j]];
            next[v] = base + dterm + DAMPING * sum;
            diff += fabsf(next[v] - rank[v]);
        }

        float *t = rank; rank = next; next = t;      // swap pointers, no copy
        if (fixed <= 0 && diff < TOL)
            break;
    }
    if (iter > maxit) iter = maxit;
    // after an odd number of swaps the result is in the buffer we malloc'd
    if (iter % 2 == 1) { memcpy(next, rank, V * sizeof(float)); free(rank); }
    else free(next);
    free(contrib);
    return iter;
}

#ifndef CPU_ONLY
// ---------------------------------------------------------------------------
// GPU kernels
// ---------------------------------------------------------------------------

// Block-level sum in shared memory (reduction from the lectures, K2 style:
// active threads stay contiguous), then ONE atomicAdd per block (privatization).
__device__ void block_sum_atomic(double val, double *out)
{
    __shared__ double s[BLOCK_SIZE];
    s[threadIdx.x] = val;
    __syncthreads();
    for (int stride = blockDim.x / 2; stride > 0; stride >>= 1) {
        if (threadIdx.x < stride)
            s[threadIdx.x] += s[threadIdx.x + stride];
        __syncthreads();
    }
    if (threadIdx.x == 0)
        atomicAdd(out, s[0]);       // double atomicAdd needs sm_60+
}

// Step 1: map contrib[u] = rank[u]/out_deg[u], and reduce the dangling mass.
__global__ void contribKernel(const float *rank, const int *out_deg, float *contrib,
                              double *dangling, int V)
{
    int u = blockIdx.x * blockDim.x + threadIdx.x;
    double d = 0.0;
    if (u < V) {
        int k = out_deg[u];
        if (k > 0) contrib[u] = rank[u] / k;
        else { contrib[u] = 0.0f; d = rank[u]; }
    }
    block_sum_atomic(d, dangling);
}

// Step 2 (version A, "pull"): one thread per page v, sums its incoming links.
// Each thread writes only next[v], so there are no races and no atomics.
__global__ void pullKernel(const int *row_ptr, const int *col_idx, const float *contrib,
                           const float *rank, float *next, const double *dangling,
                           double *diff, int V)
{
    int v = blockIdx.x * blockDim.x + threadIdx.x;
    double change = 0.0;
    if (v < V) {
        float sum = 0.0f;
        for (int j = row_ptr[v]; j < row_ptr[v + 1]; j++)
            sum += contrib[col_idx[j]];
        float base = (1.0f - DAMPING) / V + DAMPING * (float)(*dangling / V);
        float r = base + DAMPING * sum;
        next[v] = r;
        change = fabsf(r - rank[v]);
    }
    block_sum_atomic(change, diff);
}

// Step 2 (version B, "warp per page"): 32 threads share one page's incoming list,
// so a page with 10,000 in-links is not left to a single slow thread.
// Lanes read neighbouring col_idx entries -> coalesced; then a warp shuffle sum.
__global__ void warpKernel(const int *row_ptr, const int *col_idx, const float *contrib,
                           const float *rank, float *next, const double *dangling,
                           double *diff, int V)
{
    int lane = threadIdx.x & 31;
    int v = (blockIdx.x * blockDim.x + threadIdx.x) >> 5;    // one warp = one page
    double change = 0.0;
    if (v < V) {
        float sum = 0.0f;
        for (int j = row_ptr[v] + lane; j < row_ptr[v + 1]; j += 32)
            sum += contrib[col_idx[j]];
        for (int off = 16; off > 0; off >>= 1)                // tree inside the warp
            sum += __shfl_down_sync(0xffffffff, sum, off);
        if (lane == 0) {
            float base = (1.0f - DAMPING) / V + DAMPING * (float)(*dangling / V);
            float r = base + DAMPING * sum;
            next[v] = r;
            change = fabsf(r - rank[v]);
        }
    }
    block_sum_atomic(change, diff);
}

// Step 2 (version C, "push"): one thread per LINK, adds its contribution to the
// target with atomicAdd. Perfect load balance, but popular pages are hit by many
// threads at once -> atomic contention (the histogram problem from the lectures).
__global__ void pushKernel(const int *edge_dst, const int *col_idx, const float *contrib,
                           float *sum, int E)
{
    size_t stride = (size_t)blockDim.x * gridDim.x;
    for (size_t j = (size_t)blockIdx.x * blockDim.x + threadIdx.x; j < (size_t)E; j += stride)
        atomicAdd(&sum[edge_dst[j]], contrib[col_idx[j]]);
}

// Finish for push: new rank from the summed contributions.
__global__ void applyKernel(const float *sum, const float *rank, float *next,
                            const double *dangling, double *diff, int V)
{
    int v = blockIdx.x * blockDim.x + threadIdx.x;
    double change = 0.0;
    if (v < V) {
        float base = (1.0f - DAMPING) / V + DAMPING * (float)(*dangling / V);
        float r = base + DAMPING * sum[v];
        next[v] = r;
        change = fabsf(r - rank[v]);
    }
    block_sum_atomic(change, diff);
}

__global__ void initKernel(float *rank, int V)
{
    int v = blockIdx.x * blockDim.x + threadIdx.x;
    if (v < V) rank[v] = 1.0f / V;
}

// ---------------------------------------------------------------------------
// GPU driver: the graph is copied once and stays on the GPU for all iterations.
// Only 8 bytes (the L1 change) come back per iteration, for the stop test.
// ---------------------------------------------------------------------------
enum { PULL = 0, WARP = 1, PUSH = 2 };
static const char *NAMES[] = { "pull (thread/page)", "warp/page", "push (atomic)" };

static int pagerank_gpu(const Graph *g, float *h_rank, int version, int fixed,
                        float *kernel_ms, float *total_ms)
{
    int V = g->V, E = g->E;
    cudaEvent_t t0, t1, k0, k1;
    CHECK(cudaEventCreate(&t0)); CHECK(cudaEventCreate(&t1));
    CHECK(cudaEventCreate(&k0)); CHECK(cudaEventCreate(&k1));
    CHECK(cudaEventRecord(t0));

    int *d_row, *d_col, *d_deg, *d_dst = NULL;
    float *d_rank, *d_next, *d_contrib, *d_sum = NULL;
    double *d_dangling, *d_diff;
    CHECK(cudaMalloc(&d_row, (size_t)(V + 1) * sizeof(int)));
    CHECK(cudaMalloc(&d_col, (size_t)E * sizeof(int)));
    CHECK(cudaMalloc(&d_deg, (size_t)V * sizeof(int)));
    CHECK(cudaMalloc(&d_rank, (size_t)V * sizeof(float)));
    CHECK(cudaMalloc(&d_next, (size_t)V * sizeof(float)));
    CHECK(cudaMalloc(&d_contrib, (size_t)V * sizeof(float)));
    CHECK(cudaMalloc(&d_dangling, sizeof(double)));
    CHECK(cudaMalloc(&d_diff, sizeof(double)));

    CHECK(cudaMemcpy(d_row, g->row_ptr, (size_t)(V + 1) * sizeof(int), cudaMemcpyHostToDevice));
    CHECK(cudaMemcpy(d_col, g->col_idx, (size_t)E * sizeof(int), cudaMemcpyHostToDevice));
    CHECK(cudaMemcpy(d_deg, g->out_deg, (size_t)V * sizeof(int), cudaMemcpyHostToDevice));

    if (version == PUSH) {
        // push needs the target of every link: expand row_ptr into one entry per link
        int *h_dst = (int *)malloc((size_t)E * sizeof(int));
        for (int v = 0; v < V; v++)
            for (int j = g->row_ptr[v]; j < g->row_ptr[v + 1]; j++)
                h_dst[j] = v;
        CHECK(cudaMalloc(&d_dst, (size_t)E * sizeof(int)));
        CHECK(cudaMalloc(&d_sum, (size_t)V * sizeof(float)));
        CHECK(cudaMemcpy(d_dst, h_dst, (size_t)E * sizeof(int), cudaMemcpyHostToDevice));
        free(h_dst);
    }

    int grid = (V + BLOCK_SIZE - 1) / BLOCK_SIZE;
    int warpGrid = (int)(((size_t)V * 32 + BLOCK_SIZE - 1) / BLOCK_SIZE);
    int edgeGrid = (E + BLOCK_SIZE - 1) / BLOCK_SIZE;
    if (edgeGrid > 1024) edgeGrid = 1024;
    if (edgeGrid < 1) edgeGrid = 1;

    CHECK(cudaEventRecord(k0));
    initKernel<<<grid, BLOCK_SIZE>>>(d_rank, V);

    int maxit = fixed > 0 ? fixed : MAX_ITER;
    int iter;
    double h_diff = 0.0;
    for (iter = 1; iter <= maxit; iter++) {
        CHECK(cudaMemset(d_dangling, 0, sizeof(double)));
        CHECK(cudaMemset(d_diff, 0, sizeof(double)));
        contribKernel<<<grid, BLOCK_SIZE>>>(d_rank, d_deg, d_contrib, d_dangling, V);

        if (version == PULL)
            pullKernel<<<grid, BLOCK_SIZE>>>(d_row, d_col, d_contrib, d_rank, d_next,
                                             d_dangling, d_diff, V);
        else if (version == WARP)
            warpKernel<<<warpGrid, BLOCK_SIZE>>>(d_row, d_col, d_contrib, d_rank, d_next,
                                                 d_dangling, d_diff, V);
        else {
            CHECK(cudaMemset(d_sum, 0, (size_t)V * sizeof(float)));
            pushKernel<<<edgeGrid, BLOCK_SIZE>>>(d_dst, d_col, d_contrib, d_sum, E);
            applyKernel<<<grid, BLOCK_SIZE>>>(d_sum, d_rank, d_next, d_dangling, d_diff, V);
        }
        CHECK(cudaGetLastError());

        // 8-byte copy back; this also waits for the kernels to finish
        CHECK(cudaMemcpy(&h_diff, d_diff, sizeof(double), cudaMemcpyDeviceToHost));
        float *t = d_rank; d_rank = d_next; d_next = t;
        if (fixed <= 0 && h_diff < TOL)
            break;
    }
    if (iter > maxit) iter = maxit;
    CHECK(cudaEventRecord(k1));

    CHECK(cudaMemcpy(h_rank, d_rank, (size_t)V * sizeof(float), cudaMemcpyDeviceToHost));
    CHECK(cudaFree(d_row)); CHECK(cudaFree(d_col)); CHECK(cudaFree(d_deg));
    CHECK(cudaFree(d_rank)); CHECK(cudaFree(d_next)); CHECK(cudaFree(d_contrib));
    CHECK(cudaFree(d_dangling)); CHECK(cudaFree(d_diff));
    if (d_dst) { CHECK(cudaFree(d_dst)); CHECK(cudaFree(d_sum)); }
    CHECK(cudaEventRecord(t1));
    CHECK(cudaEventSynchronize(t1));

    CHECK(cudaEventElapsedTime(kernel_ms, k0, k1));   // iterations only
    CHECK(cudaEventElapsedTime(total_ms, t0, t1));    // + malloc, copies, free
    cudaEventDestroy(t0); cudaEventDestroy(t1); cudaEventDestroy(k0); cudaEventDestroy(k1);
    return iter;
}
#endif

// ---------------------------------------------------------------------------
// Checking and printing
// ---------------------------------------------------------------------------
static void top_k(const float *rank, int V, int *idx, int k)
{
    // simple selection: k passes over V, fine for k = 20
    for (int i = 0; i < k; i++) {
        int best = -1;
        for (int v = 0; v < V; v++) {
            int taken = 0;
            for (int j = 0; j < i; j++) if (idx[j] == v) { taken = 1; break; }
            if (!taken && (best < 0 || rank[v] > rank[best])) best = v;
        }
        idx[i] = best;
    }
}

static char **load_titles(const char *path, int V)
{
    FILE *f = fopen(path, "r");
    if (!f) return NULL;
    char **t = (char **)calloc(V, sizeof(char *));
    char line[1024];
    for (int i = 0; i < V && fgets(line, sizeof line, f); i++) {
        line[strcspn(line, "\n")] = 0;
        t[i] = strdup(line);
    }
    fclose(f);
    return t;
}

int main(int argc, char **argv)
{
    if (argc < 2) {
        printf("Usage: %s graph.bin [titles.txt | -] [fixed_iters]\n", argv[0]);
        return 1;
    }
    Graph g = load_graph(argv[1]);
    int fixed = argc >= 4 ? atoi(argv[3]) : 0;
    int V = g.V, k = V < TOP_K ? V : TOP_K;
    int dangling = 0, maxin = 0;
    for (int v = 0; v < V; v++) {
        if (g.out_deg[v] == 0) dangling++;
        int in = g.row_ptr[v + 1] - g.row_ptr[v];
        if (in > maxin) maxin = in;
    }
    printf("graph %s: V=%d E=%d dangling=%d max in-degree=%d\n",
           argv[1], V, g.E, dangling, maxin);

    // CPU
    float *cpu = (float *)malloc(V * sizeof(float));
    double c0 = now_ms();
    int cpu_iter = pagerank_cpu(&g, cpu, fixed);
    double cpu_ms = now_ms() - c0;
    printf("CPU                 : %3d iterations  %10.2f ms\n", cpu_iter, cpu_ms);

    int cpu_top[TOP_K];
    top_k(cpu, V, cpu_top, k);

#ifndef CPU_ONLY
    float *gpu = (float *)malloc(V * sizeof(float));
    float kms[3], tms[3];
    int ok_all = 1;
    for (int ver = 0; ver < 3; ver++) {
        float kernel_ms, total_ms;
        pagerank_gpu(&g, gpu, ver, fixed, &kernel_ms, &total_ms);   // warm-up (CUDA context, caches)
        int it = pagerank_gpu(&g, gpu, ver, fixed, &kernel_ms, &total_ms);
        kms[ver] = kernel_ms; tms[ver] = total_ms;

        // correctness: largest RELATIVE per-page difference, and the same top 20.
        // Float sums depend on the order of additions, so CPU and GPU cannot be
        // bit-identical; 1e-3 relative is far below what changes the ranking.
        double maxerr = 0.0;
        for (int v = 0; v < V; v++) {
            double e = fabs((double)gpu[v] - cpu[v]) / cpu[v];
            if (e > maxerr) maxerr = e;
        }
        int gpu_top[TOP_K];
        top_k(gpu, V, gpu_top, k);
        // same top 20, in the same order; two pages may swap only if their CPU
        // ranks are a tie (differ by < 0.01%), because float order can flip a tie
        int same = 1, ties = 0;
        for (int i = 0; i < k; i++) {
            if (gpu_top[i] == cpu_top[i]) continue;
            double a = cpu[gpu_top[i]], b = cpu[cpu_top[i]];
            if (fabs(a - b) / b < 1e-4) ties++;
            else same = 0;
        }
        int ok = maxerr < 1e-3 && same;
        ok_all &= ok;
        printf("GPU %-16s: %3d iterations  %10.2f ms kernels  %10.2f ms total  "
               "speedup %6.2fx (%5.2fx incl. copies)  max rel err %.1e  top%d %s\n",
               NAMES[ver], it, kernel_ms, total_ms, cpu_ms / kernel_ms, cpu_ms / total_ms,
               maxerr, k, !same ? "DIFFERENT" : ties ? "same (tie swapped)" : "same");
    }
    printf("check: %s\n", ok_all ? "PASS" : "FAIL");
    // one machine-readable line for the notebook
    printf("RESULT,%s,%d,%d,%d,%.3f,%.3f,%.3f,%.3f,%.3f,%.3f,%.3f\n", argv[1], V, g.E, cpu_iter, cpu_ms,
           kms[0], tms[0], kms[1], tms[1], kms[2], tms[2]);
    free(gpu);
#endif

    if (argc >= 3 && strcmp(argv[2], "-") != 0) {
        char **titles = load_titles(argv[2], V);
        if (titles) {
            printf("\nTop %d pages (CPU ranks):\n", k);
            for (int i = 0; i < k; i++)
                printf("%2d. %.6f  %s\n", i + 1, cpu[cpu_top[i]],
                       titles[cpu_top[i]] ? titles[cpu_top[i]] : "?");
        }
    }
    free(cpu); free(g.row_ptr); free(g.col_idx); free(g.out_deg);
    return 0;
}

## 2. Optional: cache data in Google Drive
Colab deletes files when the session ends. Set `USE_DRIVE = True` to keep the
converted graph in `MyDrive/pagerank-thwiki/` and skip the download next time.

In [ ]:
USE_DRIVE = False
import os, shutil
CACHE = None
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE = '/content/drive/MyDrive/pagerank-thwiki'
    os.makedirs(CACHE + '/data', exist_ok=True)
    for f in os.listdir(CACHE + '/data'):
        shutil.copy(f'{CACHE}/data/{f}', 'data/')
print('cached files:', sorted(os.listdir('data')))

## 3. Download the dump and build the graph
`linktarget` only exists in newer dumps; if it fails to download, the script
uses the older pagelinks format automatically.

In [ ]:
import os
if not os.path.exists('data/graph_100.bin'):
    for t in ['page', 'redirect', 'pagelinks', 'linktarget']:
        os.system(f'wget -q -nc -P dump https://dumps.wikimedia.org/thwiki/latest/thwiki-latest-{t}.sql.gz')
    !ls -lh dump
    !python3 prep/convert.py --dump dump --out data
    if CACHE:
        !cp data/* {CACHE}/data/
else:
    print('graph already built')
!ls -lh data

## 4. Compile
`-arch=sm_75` matches the T4 (double-precision `atomicAdd` needs sm_60 or newer).

In [ ]:
!nvcc -O3 -arch=sm_75 -o pagerank pagerank.cu && echo compiled

## 5. Correctness: full graph, run until converged
Checks every GPU version against the CPU: max relative error per page and the
same top-20 pages. Then prints the 20 highest-ranked Thai Wikipedia articles.

In [ ]:
!./pagerank data/graph_100.bin data/titles.txt | tee results/correctness.txt

## 6. Benchmark
Fixed 50 iterations so CPU and GPU do exactly the same work. Each GPU version is
run once as warm-up and then timed. Repeated `REPEATS` times; the median is used.

In [ ]:
import subprocess, pandas as pd
REPEATS, ITERS = 3, 50
cols = ['file','V','E','iters','cpu_ms','pull_kernel_ms','pull_total_ms',
        'warp_kernel_ms','warp_total_ms','push_kernel_ms','push_total_ms']
rows = []
for pct in [10, 25, 50, 100]:
    for r in range(REPEATS):
        out = subprocess.run(['./pagerank', f'data/graph_{pct}.bin', '-', str(ITERS)],
                             capture_output=True, text=True).stdout
        print(out)
        line = [l for l in out.splitlines() if l.startswith('RESULT,')][0]
        vals = line.split(',')[1:]
        row = dict(zip(cols, vals)); row['pct'] = pct
        rows.append(row)
df = pd.DataFrame(rows)
for c in cols[1:]:
    df[c] = pd.to_numeric(df[c])
res = df.groupby('pct').median(numeric_only=True).reset_index()
for v in ['pull', 'warp', 'push']:
    res[f'{v}_speedup'] = res.cpu_ms / res[f'{v}_kernel_ms']
    res[f'{v}_speedup_total'] = res.cpu_ms / res[f'{v}_total_ms']
gpu = subprocess.run(['nvidia-smi','--query-gpu=name','--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip()
res['gpu'] = gpu
res.to_csv('results/benchmark.csv', index=False)
pd.set_option('display.width', 200)
res

## 7. Charts for the slides

In [ ]:
import matplotlib.pyplot as plt, numpy as np
x = np.arange(len(res)); w = 0.25
labels = [f"{p}%\n{int(e/1e6*10)/10}M links" for p, e in zip(res.pct, res.E)]

fig, ax = plt.subplots(figsize=(8, 4.5))
for i, (v, name) in enumerate([('pull','pull (thread/page)'), ('warp','warp/page'), ('push','push (atomic)')]):
    b = ax.bar(x + (i-1)*w, res[f'{v}_speedup'], w, label=name)
    ax.bar_label(b, fmt='%.1fx', fontsize=8)
ax.set_xticks(x, labels); ax.set_ylabel('speedup over CPU (kernels only)')
ax.set_title(f'GPU speedup vs graph size ({gpu}, {ITERS} iterations)')
ax.legend(); ax.spines[['top','right']].set_visible(False)
plt.tight_layout(); plt.savefig('results/speedup.png', dpi=200); plt.show()

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(res.E/1e6, res.cpu_ms, 'o-', label='CPU')
for v, name in [('pull','pull'), ('warp','warp/page'), ('push','push')]:
    ax.plot(res.E/1e6, res[f'{v}_kernel_ms'], 'o-', label=f'GPU {name}')
ax.set_xlabel('links (millions)'); ax.set_ylabel(f'time for {ITERS} iterations (ms)')
ax.set_yscale('log'); ax.set_title('Time vs graph size'); ax.legend()
ax.spines[['top','right']].set_visible(False)
plt.tight_layout(); plt.savefig('results/time.png', dpi=200); plt.show()

## 8. Save results
Download `results/` (or it is copied to Drive if `USE_DRIVE = True`) and send
`benchmark.csv`, `correctness.txt` and the two PNGs back for the slides.

In [ ]:
!ls -lh results
if CACHE:
    !cp -r results {CACHE}/
else:
    from google.colab import files
    !zip -q -r results.zip results
    files.download('results.zip')